In [1]:
pip install torch transformers datasets peft accelerate bitsandbytes

In [2]:
pip install -U "huggingface_hub[cli]"

In [ ]:
!hf auth login --token <YOUR_HUGGINGFACE_TOKEN>

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `hf_3` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `hf_3`


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [15]:
from datasets import load_dataset
dataset=load_dataset(
    "json",
    data_files={
        "train":"train.jsonl",
        "validation":"validation.jsonl",
        "test":"test.jsonl"
    }
)

train_dataset = dataset["train"]
val_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [6]:
def preprocess(example):
    # Convert chat messages to Gemma chat format
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length"
    )

    # Labels for causal language modeling
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

In [7]:
train_dataset = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    preprocess,
    remove_columns=val_dataset.column_names
)

test_dataset = test_dataset.map(
    preprocess,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/1217 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Map:   0%|          | 0/153 [00:00<?, ? examples/s]

In [8]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)



In [9]:
!pip install -U trl

In [11]:
import trl
print(trl.__version__)

1.8.0


In [14]:
print(train_dataset)
print(train_dataset.features)
print(train_dataset[0])

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1217
})
{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}
{'input_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [20]:
from trl import SFTTrainer, SFTConfig

model.gradient_checkpointing_enable()
model.config.use_cache = False

training_args = SFTConfig(
    output_dir="./finetuned-llm",
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_steps=100,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.train()

Step,Training Loss
10,3.780051
20,2.557301
30,2.254367
40,2.117522
50,2.040915
60,1.962795
70,1.889074
80,1.880857
90,1.898345
100,1.834141


TrainOutput(global_step=231, training_loss=1.943325552073392, metrics={'train_runtime': 737.0328, 'train_samples_per_second': 4.954, 'train_steps_per_second': 0.313, 'total_flos': 3359491505187840.0, 'train_loss': 1.943325552073392, 'entropy': 1.6037282943725586, 'num_tokens': 512715.0, 'mean_token_accuracy': 0.6666666865348816, 'epoch': 3.0})

In [28]:
print(type(model))

<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM'>


In [21]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("./gemma-therapist")
tokenizer.save_pretrained("./gemma-therapist")

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gemma-therapist/tokenizer_config.json',
 './gemma-therapist/chat_template.jinja',
 './gemma-therapist/tokenizer.json')

In [23]:
!zip -r gemma-therapist.zip gemma-therapist

  adding: gemma-therapist/ (stored 0%)
  adding: gemma-therapist/tokenizer.json (deflated 83%)
  adding: gemma-therapist/tokenizer_config.json (deflated 60%)
  adding: gemma-therapist/config.json (deflated 69%)
  adding: gemma-therapist/chat_template.jinja (deflated 70%)
  adding: gemma-therapist/generation_config.json (deflated 36%)
  adding: gemma-therapist/model.safetensors (deflated 16%)


In [27]:
messages = [
    {"role": "user", "content": "I feel anxious all the time. What should I do?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print(response)

Okay, it’s really brave of you to recognize that you're feeling anxious constantly – that’s a huge first step! It sounds like you're experiencing a lot of distress and potentially some underlying issues. Here’s a breakdown of what might be happening and what steps you can take, broken down into different areas:

**1. Understanding Your Anxiety:**

* **What kind of anxiety is it?** Anxiety manifests differently for everyone. Is it generalized anxiety (feeling worried about many things), social anxiety (fear of judgment in specific situations), panic attacks, or something else? Knowing the type of anxiety helps determine the best treatment approach.
* **Recognize the physical symptoms:** Anxious feelings often translate into physical sensations like racing heart, sweating, shortness of breath, trembling, nausea, headaches, muscle tension, fatigue, sleep problems, changes in appetite, etc.  Understanding these signals can help you catch them before they become overwhelming.
* **Identify y

In [24]:
from google.colab import files

files.download("gemma-therapist.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "./gemma-therapist"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model.eval()

print("Model loaded successfully!")


def ask(question: str):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    return response


while True:
    question = input("\nYou: ")

    if question.lower() in ["exit", "quit"]:
        break

    answer = ask(question)

    print("\nTherapist:", answer)

Loading model...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded successfully!

You: i am not having a good day

Therapist: I'm sincerely sorry to hear you’re having a difficult day. It takes courage to admit that, and I want you to know that your feelings are valid. There is absolutely nothing wrong with you for feeling this way.

Let’s see if we can figure out what might be going on and find some ways to help you feel a little better.  Would you like to tell me a bit about what's making today difficult? No pressure at all – just whatever comes to mind.stony so don't feel obligated to give me details if you don't want them. 

Here are some things that could help, but please only share what feels comfortable for you:

*   What happened today specifically?
*   How would you describe how you're feeling (e.g., sad, angry, anxious)?
*   Is there anything in particular that triggered these feelings?stonyincase it's a big issuecopycopycopy
Okay, let’s start with the first question - what happened today specifically?
copycopy
Take a breath bef

KeyboardInterrupt: Interrupted by user

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [6]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [7]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "./finetuned-llm/checkpoint-231"
)

merged_model = model.merge_and_unload()

merged_model.save_pretrained("./gemma-therapist-full")
tokenizer.save_pretrained("./gemma-therapist-full")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gemma-therapist-full/tokenizer_config.json',
 './gemma-therapist-full/chat_template.jinja',
 './gemma-therapist-full/tokenizer.json')

In [8]:
import os

size = os.path.getsize("./gemma-therapist-full/model.safetensors") / (1024**3)
print(f"{size:.2f} GB")

1.86 GB


In [9]:
import json

with open("./gemma-therapist-full/config.json") as f:
    config = json.load(f)

print("quantization_config" in config)

False


In [10]:
!zip -r gemma-therapist-full.zip gemma-therapist-full

  adding: gemma-therapist-full/ (stored 0%)
  adding: gemma-therapist-full/tokenizer.json (deflated 83%)
  adding: gemma-therapist-full/tokenizer_config.json (deflated 60%)
  adding: gemma-therapist-full/config.json (deflated 70%)
  adding: gemma-therapist-full/chat_template.jinja (deflated 70%)
  adding: gemma-therapist-full/generation_config.json (deflated 34%)
  adding: gemma-therapist-full/model.safetensors (deflated 22%)


In [11]:
from google.colab import files
files.download("gemma-therapist-full.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>